In [ ]:
import re
from datetime import datetime
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from duckduckgo_search import DDGS

from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing import Annotated
from typing_extensions import TypedDict

# =====================================
# Config
# =====================================
OPENROUTER_KEY = "YOUR_OPENROUTER_API_KEY"
PDF_PATH ="D:\\D\\work\\code\\2005.11401v4.pdf"

llm = ChatOpenAI(
    model="openai/gpt-oss-120b:free",
    api_key=OPENROUTER_KEY,
    base_url="https://openrouter.ai/api/v1"
)

# =====================================
# PDF Load (One time)
# =====================================
try:
    chunks = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=50).split_documents(
        PyPDFLoader(PDF_PATH).load()
    )
    vectorstore = FAISS.from_documents(chunks, HuggingFaceEmbeddings())
    print("✅ PDF Ready")
except:
    vectorstore = None
    print("⚠️ PDF nahi mili")

# =====================================
# Tools with @tool decorator
# =====================================

@tool
def pdf_search(query: str) -> str:
    """Search information from loaded PDF document"""
    if not vectorstore:
        return "PDF available nahi hai"
    docs = vectorstore.similarity_search(query, k=3)
    return "\n\n".join([d.page_content for d in docs])


@tool
def web_search(query: str) -> str:
    """Search latest information from internet"""
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=3))
            return "\n".join([r["body"] for r in results])
    except:
        return "Web search failed"


@tool
def calculator(expression: str) -> str:
    """Calculate math expressions like 2+2, 10*5, 100/4"""
    try:
        clean = re.findall(r"[\d\+\-\*\/\(\)\. ]+", expression)[0]
        return f"= {eval(clean)}"
    except:
        return "Invalid expression"


@tool
def current_time(query: str) -> str:
    """Get current date and time"""
    return datetime.now().strftime("%d %B %Y, %H:%M:%S")


tools = [pdf_search, web_search, calculator, current_time]

# =====================================
# LLM with tools
# =====================================
llm_with_tools = llm.bind_tools(tools)

# =====================================
# State
# =====================================
class State(TypedDict):
    messages: Annotated[list, add_messages]

# =====================================
# Agent Node
# =====================================
def agent(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

# =====================================
# Build Graph
# =====================================
graph = StateGraph(State)
graph.add_node("agent", agent)
graph.add_node("tools", ToolNode(tools))
graph.set_entry_point("agent")
graph.add_conditional_edges("agent", tools_condition)
graph.add_edge("tools", "agent")

app = graph.compile()

# =====================================
# Chat
# =====================================
history = [SystemMessage(content="""You are a helpful AI assistant with these tools:
- pdf_search: Search PDF content
- web_search: Search internet
- calculator: Do math
- current_time: Get time/date
Use tools when needed.""")]

print("\n🤖 AI Agent Ready! (exit to quit)\n")

while True:
    user = input("You: ").strip()
    
    if user.lower() in ["exit", "quit"]:
        break 
    
    history.append(HumanMessage(content=user))
    
    result = app.invoke({"messages": history})
    
    # Get last AI message
    ai_msg = [m for m in result["messages"] if hasattr(m, "content") and m.content][-1]
    
    history.append(ai_msg)
    
    print(f"\n🤖 {ai_msg.content}\n")

C:\Users\Ali Mehdi\AppData\Local\Temp\ipykernel_4180\340263747.py:37: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  vectorstore = FAISS.from_documents(chunks, HuggingFaceEmbeddings())
C:\Users\Ali Mehdi\AppData\Local\Temp\ipykernel_4180\340263747.py:37: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  vectorstore = FAISS.from_documents(chunks, HuggingFaceEmbeddings())


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\Ali Mehdi\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Ali Mehdi\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ PDF Ready

🤖 AI Agent Ready! (exit to quit)


🤖 **Artificial Intelligence (AI) क्या है?**

Artificial Intelligence, यानी **कृत्रिम बुद्धिमत्ता**, वह तकनीक है जो कंप्यूटर सिस्टम को मानव‑समानों सोच‑और‑समझ क्षमताएँ देने का प्रयास करती है। सरल शब्दों में, AI उन प्रोग्रामों और एल्गोरिद्मों को कहते हैं जो:

| मुख्य क्षमता | उदाहरण |
|-------------|----------|
| **सीखना (Learning)** | मशीन लर्निंग मॉडल को डेटा से पैटर्न सीखना (जैसे, फोटो में बिल्ली‑कुत्ते की पहचान) |
| **तर्क करना (Reasoning)** | नियम‑आधारित सिस्टम से निष्कर्ष निकालना (जैसे, डॉक्टर की सलाह देना) |
| **समझना (Understanding)** | प्राकृतिक भाषा समझना, आवाज़ पहचानना (जैसे, Siri, Alexa) |
| **निर्णय लेना (Decision‑making)** | गेम खेलने में रणनीति बनाना (जैसे, Chess‑एंजिन, Go‑एंजिन) |
| **सर्जनात्मक कार्य (Creativity)** | नई संगीत रचना, चित्र बनाना, लेख लिखना (जैसे, DALL·E, ChatGPT) |

### AI के प्रमुख प्रकार

| प्रकार | विवरण | सामान्य उपयोग |
|--------|--------|----------------|
| **Narrow AI (संकुचित AI)** | एक ही कार्य में वि

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

OPENROUTER_KEY = "apni_key_yahan"

llm = ChatOpenAI(
    model="meta-llama/llama-3.3-70b-instruct:free",  # sahi naam
    api_key=OPENROUTER_KEY,
    base_url="https://openrouter.ai/api/v1"
)

@tool
def calculator(expression: str) -> str:
    """Math calculations"""
    try:
        return str(eval(expression))
    except:
        return "Invalid"

tools = [calculator]
agent = create_react_agent(llm, tools)

C:\Users\Ali Mehdi\AppData\Local\Temp\ipykernel_8856\1202776052.py:22: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools)
